In [ ]:
import nltk
nltk.download('punkt_tab')      
nltk.download('wordnet')    
nltk.download('omw-1.4') 
nltk.download('averaged_perceptron_tagger_eng') 

file_path = "../test_files/test_text.txt"

with open(file_path, "r") as file:
    test_text = file.read()

# https://www.geeksforgeeks.org/python/python-lemmatization-with-nltk/

from nltk.tokenize import word_tokenize
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer

from nltk.probability import FreqDist

import pandas as pd

In [ ]:


# Test for all lemmatized_frequency_distribution and clean_freq_dist from text_processing.py
test_str = test_text
from text_processing import lemmatized_frequency_distribution, clean_freq_dist

lem_freq = lemmatized_frequency_distribution(test_str)
clean_freq = clean_freq_dist(lem_freq)

print("Lemmatized Frequency Distribution:", lem_freq)
print("Clean Frequency Distribution:", clean_freq)



In [ ]:
test_str = "The running boy ran his daily run because he runs fast."
test_str = test_text

# Tokenize the string by word (break it into the individual words)
tokenized_str = word_tokenize(test_str)
tokenized_str


In [ ]:
# Lemmatize the string, reduce words to base form

lemmatizer = WordNetLemmatizer()
tagged_tokenized_str = pos_tag(tokenized_str)

# todo: use:
# wordnet.NOUN = 'n'
# wordnet.VERB = 'v'
# wordnet.ADJ  = 'a'
# wordnet.ADV  = 'r'
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('V'):
        return 'v'
    elif tag.startswith('N'):
        return 'n'
    elif tag.startswith('R'):
        return 'r'
    else:
        return 'n'


lemmatized_str = []
for word, tag in tagged_tokenized_str:
    if word.lower() == 'are' or word.lower() in ['is', 'am']:
        lemmatized_str.append(word)
    else:
        lemmatized_str.append(
            lemmatizer.lemmatize(word, get_wordnet_pos(tag)))
			
lemmatized_str




In [ ]:
# Make frequency distribution

fdist = FreqDist()
for word in lemmatized_str:
	fdist[word.lower()] += 1

fdist

## Write fdist to csv file
import csv
with open('./test_files/fdist.csv', 'w', newline='') as csvfile:
		writer = csv.writer(csvfile)
		writer.writerow(['Word', 'Frequency'])
		for word, frequency in fdist.items():
				writer.writerow([word, frequency])


In [ ]:
import pandas as pd

df = pd.read_csv("test_files/fdist.csv")

# Drop rows with non letter values
df = df[df['Word'].str.contains(r'[a-zA-Z]')]

# Drop rows with to be verbs
to_be_verbs = ['am', 'is', 'are', 'was', 'were', 'be', 'being', 'been']
df = df[~df['Word'].isin(to_be_verbs)]

# Drop rows with prepositions without using regex
prepositions = ['a', 'an', 'the', 'to', 'of', 'and', 'or', 'in', 'on', 'at', 'by', 'for', 'with', 'from', 'into', 'out', 'over', 'up', 'down', 'off', 'about', 'after', 'against', 'along', 'among', 'around', 'as', 'at', 'before', 'behind', 'below', 'beneath', 'beside', 'between', 'beyond', 'but', 'by', 'down', 'during', 'except', 'for', 'from', 'in', 'inside', 'into', 'like', 'minus', 'near', 'next', 'of', 'off', 'on', 'onto', 'opposite', 'out', 'outside', 'over', 'past', 'per', 'plus', 'round', 'save', 'since', 'than', 'through', 'throughout', 'till', 'times', 'to', 'toward', 'towards', 'under', 'until', 'up', 'upon', 'via', 'with', 'within', 'without', 'yet']
df = df[~df['Word'].isin(prepositions)]

# Drop rows with conjunctions
conjunctions = ['and', 'or', 'but', 'nor', 'for', 'yet', 'so', 'if', 'while', 'as', 'because', 'since', 'unless', 'once', 'when', 'till', 'unless', 'although', 'whereas', 'whether', 'how', 'although', 'whereas', 'whether', 'how', 'once', 'after', 'before', 'over', 'under', 'above', 'below', 'among', 'amid', 'amongst', 'among', 'around', 'as', 'at', 'atop', 'by', 'down', 'during', 'except', 'for', 'from', 'in', 'into', 'like', 'minus', 'near', 'next', 'of', 'off', 'on', 'onto', 'opposite', 'out', 'outside', 'over', 'past', 'per', 'plus', 'round', 'save', 'since', 'than', 'through', 'throughout', 'till', 'times', 'to', 'toward', 'towards', 'under', 'until', 'up', 'upon', 'via', 'with', 'within', 'without', 'yet']
df = df[~df['Word'].isin(conjunctions)]

# Drop rows with articles
articles = ['a', 'an', 'the']
df = df[~df['Word'].isin(articles)]

# Drop rows with pronouns
pronouns = ['i', 'you', 'he', 'she', 'it', 'we', 'they', 'me', 'my', 'him', 'her', 'us', 'mine', 'our', 'his', 'hers', 'its', 'ours', 'yours', 'theirs', 'myself', 'yourself', 'himself', 'herself', 'itself', 'yourselves', 'themselves', 'my', 'your', 'yours', 'ours', 'theirs', 'its', 'whose']
df = df[~df['Word'].isin(pronouns)]

# Sort by frequency
df = df.sort_values(by=['Frequency'], ascending=False)

df.head(10) 

In [ ]:
# write first 50 rows from df to a md file

df_50 = df.head(50)

# Convert to Markdown string
markdown_table = df_50.to_markdown(index=False)
# Write to a text file
with open('./test_files/freq_table_prompt.md', 'w') as file:
    file.write("Here is the word frequency table:\n\n")
    file.write(markdown_table)